# SECOM EDA — 공정 센서 데이터 탐색

반도체 공정 센서 590개 + 불량 라벨.
목표: 불량과 연관된 핵심 센서 변수를 찾고, SHAP 기반 인과 분석 파이프라인의 기초 데이터로 활용.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 데이터 로드
features = pd.read_csv('../data/raw/secom.data', sep=' ', header=None)
labels = pd.read_csv('../data/raw/secom_labels.data', sep=' ', header=None, names=['label', 'timestamp'])

print(f'Features shape: {features.shape}')
print(f'Labels shape: {labels.shape}')
print(f'\n불량 분포:')
print(labels['label'].value_counts())
print(f'\n불량률: {(labels["label"] == 1).mean():.2%}')

In [ ]:
# 결측치 분석
missing_pct = features.isnull().mean()
print(f'전체 결측률: {features.isnull().mean().mean():.2%}')
print(f'결측 50% 이상 컬럼: {(missing_pct > 0.5).sum()}개')
print(f'결측 0% 컬럼: {(missing_pct == 0).sum()}개')

# 결측 50% 이상 컬럼 제거
valid_cols = missing_pct[missing_pct <= 0.5].index
df = features[valid_cols].copy()
df['label'] = labels['label'].values
df['timestamp'] = pd.to_datetime(labels['timestamp'])
print(f'\n유효 컬럼: {len(valid_cols)}개 / 전체 {features.shape[1]}개')
print(f'결측 제거 후 shape: {df.shape}')

In [ ]:
# 결측 대체 (중앙값)
sensor_cols = [c for c in df.columns if c not in ['label', 'timestamp']]
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

# 분산 0인 컬럼 제거 (상수 값)
zero_var = df[sensor_cols].std() == 0
drop_cols = zero_var[zero_var].index.tolist()
df = df.drop(columns=drop_cols)
sensor_cols = [c for c in df.columns if c not in ['label', 'timestamp']]
print(f'상수 컬럼 제거: {len(drop_cols)}개')
print(f'최종 센서 수: {len(sensor_cols)}개')

In [ ]:
# 불량(label=1)과 상관관계 높은 변수 Top 30
corr_with_label = df[sensor_cols].corrwith(df['label']).abs().sort_values(ascending=False)
top30 = corr_with_label.head(30)

fig, ax = plt.subplots(figsize=(10, 8))
top30.plot(kind='barh', ax=ax, color='#1565C0')
ax.set_xlabel('|상관계수|')
ax.set_title('불량(label=1)과 상관관계 Top 30 센서')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../docs/proposal/figures/secom_top30_corr.png', dpi=150)
plt.show()
print(top30)

In [ ]:
# 시계열 불량률 추이 (시간대별)
df['hour'] = df['timestamp'].dt.hour
hourly = df.groupby('hour')['label'].agg(['mean', 'count', 'sum'])
hourly.columns = ['defect_rate', 'total', 'defects']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.bar(hourly.index, hourly['defect_rate'] * 100, color='#E57373')
ax1.set_ylabel('불량률 (%)')
ax1.set_title('시간대별 불량률')

ax2.bar(hourly.index, hourly['total'], color='#90CAF9', label='전체')
ax2.bar(hourly.index, hourly['defects'], color='#E57373', label='불량')
ax2.set_xlabel('시간 (hour)')
ax2.set_ylabel('건수')
ax2.legend()

plt.tight_layout()
plt.savefig('../docs/proposal/figures/secom_hourly_defect.png', dpi=150)
plt.show()

In [ ]:
# SHAP 분석 (LightGBM)
from sklearn.model_selection import train_test_split
import shap

try:
    import lightgbm as lgb
except ImportError:
    !pip install lightgbm shap -q
    import lightgbm as lgb
    import shap

X = df[sensor_cols]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = lgb.LGBMClassifier(n_estimators=200, max_depth=5, random_state=42, verbose=-1)
model.fit(X_train, y_train)

from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# SHAP Summary Plot
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# 불량 클래스(1)에 대한 SHAP
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(sv, X_test, max_display=20, show=False)
plt.title('불량 원인 변수 SHAP 분석 (Top 20)')
plt.tight_layout()
plt.savefig('../docs/proposal/figures/secom_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 10 원인 변수
mean_shap = np.abs(sv).mean(axis=0)
top10_idx = np.argsort(mean_shap)[::-1][:10]
print('\n=== 불량 원인 Top 10 센서 ===')
for rank, idx in enumerate(top10_idx, 1):
    print(f'  {rank}. 센서 {sensor_cols[idx]} — SHAP 평균: {mean_shap[idx]:.4f}')

In [ ]:
# 전처리된 데이터 저장 (후속 파이프라인용)
df.to_csv('../data/processed/secom_cleaned.csv', index=False)
print(f'저장 완료: data/processed/secom_cleaned.csv ({df.shape})')
print(f'센서 컬럼: {len(sensor_cols)}개')
print(f'불량률: {y.mean():.2%}')